# Bühlmann-Straub Credibility Blend

Blends developed market cell-level estimates (the **collective** prior) with emerging market
experience (the **individual** data) using Bühlmann-Straub credibility theory.

**Method:**
- Common risk cells defined via harmonized features (construction_type, age_band, sum_insured_band, vehicle_value_group)
- Credibility parameters (K = process variance / structural variance) estimated from the developed market
- For each emerging cell: Z = n_emg / (n_emg + K), blended estimate = Z × emg_mean + (1-Z) × dev_mean_scaled
- Developed cell means adjusted by overall market ratio R = emg_overall / dev_overall
- Final prediction = α × credibility + (1-α) × model_blend, α optimized on 2024 holdout

**Cell definitions:**
- Motor: (driver_age_band, vehicle_value_group) — 5 × 20 = 100 cells
- Buildings: (construction_type, policyholder_age_band, sum_insured_band) — ~125 cells
- Contents: (policyholder_age_band, sum_insured_band) — ~25 cells

In [0]:
# Step 1: Load both markets + create common risk cells
import pandas as pd
import numpy as np
import os

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'
DEV_DIR = f'{BASE}/files_v6/'
EMG_DIR = f'{BASE}/files_emerging_v6/'
SUB_DIR = f'{BASE}/submissions_emerging_v6/'
os.makedirs(SUB_DIR, exist_ok=True)

# === Sum insured bands (consistent across markets) ===
SI_BANDS_BUILDINGS = [0, 100_000, 200_000, 300_000, 500_000, np.inf]
SI_BANDS_CONTENTS = [0, 20_000, 40_000, 60_000, 100_000, np.inf]
SI_LABELS = ['0-100K', '100-200K', '200-300K', '300-500K', '500K+']
SI_LABELS_C = ['0-20K', '20-40K', '40-60K', '60-100K', '100K+']

# === Load proc files ===
dev, emg = {}, {}
for line in ['motor', 'buildings', 'contents']:
    dev[line] = pd.read_parquet(os.path.join(DEV_DIR, f'proc_{line}_v6_ldf.parquet'))
    emg[line] = pd.read_parquet(os.path.join(EMG_DIR, f'proc_{line}_v6_ldf.parquet'))
    print(f'{line}: dev={dev[line].shape}, emg={emg[line].shape}')

# === Create cell keys ===
def add_cell_key(df, line):
    df = df.copy()
    if line == 'motor':
        df['cell_key'] = df['driver_age_band'].astype(str) + '|' + df['vehicle_value_group'].astype(str)
    elif line == 'buildings':
        df['si_band'] = pd.cut(df['sum_insured'], bins=SI_BANDS_BUILDINGS, labels=SI_LABELS, right=False).astype(str)
        df['cell_key'] = df['construction_type'].astype(str) + '|' + df['policyholder_age_band'].astype(str) + '|' + df['si_band']
    elif line == 'contents':
        df['si_band'] = pd.cut(df['sum_insured'], bins=SI_BANDS_CONTENTS, labels=SI_LABELS_C, right=False).astype(str)
        df['cell_key'] = df['policyholder_age_band'].astype(str) + '|' + df['si_band']
    return df

for line in ['motor', 'buildings', 'contents']:
    dev[line] = add_cell_key(dev[line], line)
    emg[line] = add_cell_key(emg[line], line)
    dev_cells = dev[line]['cell_key'].nunique()
    emg_cells = emg[line]['cell_key'].nunique()
    common_cells = len(set(dev[line]['cell_key']) & set(emg[line]['cell_key']))
    print(f'{line}: dev cells={dev_cells}, emg cells={emg_cells}, common={common_cells}')

print('\nStep 1 complete.')

In [0]:
# Step 2: Bühlmann-Straub credibility parameters + blended cell estimates
#
# For each line:
#   x_ij = ultimate_claims_j / exposure_j  (per-unit-exposure claim cost)
#   w_ij = exposure_j                       (weight)
#   X̄_i = Σ w_ij x_ij / Σ w_ij = Σ ultimate_claims / Σ exposure  (cell mean)
#   X̄ = overall mean (exposure-weighted across all cells)
#   s² = process variance = Σ_i Σ_j w_ij (x_ij - X̄_i)² / (N - I)
#   a  = structural variance = max(0, [Σ n_i (X̄_i - X̄)² - (I-1)s²] / [Σ n_i - Σ n_i²/Σ n_i])
#   K  = s² / a  (credibility constant)
#   Z_i = n_i / (n_i + K)  (credibility factor for emerging cell i)
#   Blended = Z_i × emg_cell_mean + (1-Z_i) × dev_cell_mean × R
#   where R = emg_overall_mean / dev_overall_mean (market level adjustment)

def compute_buhlmann_straub(dev_df, emg_df, line):
    """Estimate Bühlmann-Straub parameters from developed market, apply to emerging."""
    # --- Per-unit-exposure cost ---
    dev_df = dev_df.copy()
    emg_df = emg_df.copy()
    dev_df['ppu'] = dev_df['ultimate_claims'] / dev_df['exposure'].clip(lower=1e-10)  # per-unit cost
    emg_df['ppu'] = emg_df['ultimate_claims'] / emg_df['exposure'].clip(lower=1e-10)

    # --- Cell stats for developed market ---
    dev_cells = dev_df.groupby('cell_key').agg(
        dev_n=('exposure', 'sum'),
        dev_sum_uc=('ultimate_claims', 'sum'),
    ).reset_index()
    dev_cells['dev_mean'] = dev_cells['dev_sum_uc'] / dev_cells['dev_n']  # per-unit-exposure mean

    # --- Cell stats for emerging market ---
    emg_cells = emg_df.groupby('cell_key').agg(
        emg_n=('exposure', 'sum'),
        emg_sum_uc=('ultimate_claims', 'sum'),
    ).reset_index()
    emg_cells['emg_mean'] = emg_cells['emg_sum_uc'] / emg_cells['emg_n']

    # --- Overall means ---
    dev_overall = dev_df['ultimate_claims'].sum() / dev_df['exposure'].sum()
    emg_overall = emg_df['ultimate_claims'].sum() / emg_df['exposure'].sum()
    R = emg_overall / dev_overall  # market level ratio

    # === Estimate s² and a from developed market ===
    # Process variance: within-cell weighted variance
    I = len(dev_cells)
    N = len(dev_df)
    s2_num = 0.0
    for cell_key, grp in dev_df.groupby('cell_key'):
        x_bar = grp['ultimate_claims'].sum() / grp['exposure'].sum()
        s2_num += (grp['exposure'] * (grp['ppu'] - x_bar) ** 2).sum()
    s2 = s2_num / max(N - I, 1)

    # Structural variance: between-cell variance
    n_bar = dev_cells['dev_n'].mean()
    a_num = (dev_cells['dev_n'] * (dev_cells['dev_mean'] - dev_overall) ** 2).sum()
    a_num -= (I - 1) * s2
    a_den = dev_cells['dev_n'].sum() - (dev_cells['dev_n'] ** 2).sum() / dev_cells['dev_n'].sum()
    a = max(a_num / max(a_den, 1e-10), 1e-10)

    K = s2 / a
    print(f'\n=== {line.upper()} Bühlmann-Straub ===')
    print(f'  Cells: {I} (dev), {len(emg_cells)} (emg)')
    print(f'  s²={s2:.2f}, a={a:.2f}, K={K:.2f}')
    print(f'  dev_overall={dev_overall:.2f}, emg_overall={emg_overall:.2f}, R={R:.4f}')

    # === Merge cell stats ===
    merged = emg_cells.merge(dev_cells[['cell_key', 'dev_mean']], on='cell_key', how='left')
    merged['dev_mean_scaled'] = merged['dev_mean'] * R  # scale developed mean to emerging level

    # === Credibility weights ===
    merged['Z'] = merged['emg_n'] / (merged['emg_n'] + K)
    merged['Z'] = merged['Z'].clip(0, 1)

    # For cells with no developed match, use emerging mean (Z=1)
    merged.loc[merged['dev_mean'].isna(), 'Z'] = 1.0
    merged.loc[merged['dev_mean'].isna(), 'dev_mean_scaled'] = merged.loc[merged['dev_mean'].isna(), 'emg_mean']

    # === Blended estimate ===
    merged['cred_mean'] = merged['Z'] * merged['emg_mean'] + (1 - merged['Z']) * merged['dev_mean_scaled']

    # Also compute for dev-only cells (not in emerging) — needed for test lookup
    dev_only = dev_cells[~dev_cells['cell_key'].isin(emg_cells['cell_key'])].copy()
    dev_only['dev_mean_scaled'] = dev_only['dev_mean'] * R
    dev_only['Z'] = 0.0
    dev_only['emg_mean'] = dev_only['dev_mean_scaled']  # fallback
    dev_only['cred_mean'] = dev_only['dev_mean_scaled']
    dev_only['emg_n'] = 0.0
    dev_only['emg_sum_uc'] = 0.0

    all_cells = pd.concat([merged, dev_only[merged.columns]], ignore_index=True)

    print(f'  Z distribution: mean={merged["Z"].mean():.3f}, min={merged["Z"].min():.3f}, max={merged["Z"].max():.3f}')
    print(f'  Cells with Z<0.5 (strong developed prior): {(merged["Z"]<0.5).sum()}')
    print(f'  Cells with Z>=0.8 (strong emerging signal): {(merged["Z"]>=0.8).sum()}')

    return all_cells, R, K

# Compute for each line
cell_estimates = {}
market_ratios = {}
cred_K = {}
for line in ['motor', 'buildings', 'contents']:
    cells, R, K = compute_buhlmann_straub(dev[line], emg[line], line)
    cell_estimates[line] = cells
    market_ratios[line] = R
    cred_K[line] = K

print('\nStep 2 complete.')

In [0]:
# Step 3: Apply credibility to emerging test + training data + blend with model

# Tweedie D2
def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))
def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

# === Load existing model submission ===
model_sub = pd.read_csv(os.path.join(SUB_DIR, 'blend_emerging_v6.csv'))
print(f'Loaded model submission: {model_sub.shape}')

# === Load emerging test files + clean ===
DATA_EMG = f'{BASE}/data_emerging/'
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_EMG, f'test_{line}.parquet'))

customers = pd.read_parquet(os.path.join(DATA_EMG, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_EMG, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_EMG, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_EMG, 'sample_submission.parquet'))

# Clean test files (same as data prep notebook)
CONSTRUCTION_MAP_EMG = {'BRK': 'BRICK', 'STN': 'STONE', 'WOD': 'TIMBER', 'MIX': 'MIXED'}
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    if 'compulsory_excess' in df.columns:
        df.drop('compulsory_excess', axis=1, inplace=True)

# Sum insured fix on test
for line in ['buildings', 'contents']:
    df = test[line]
    if 'sum_insured' in df.columns:
        threshold = 10_000_000 if line == 'buildings' else 1_000_000
        flagged = df['sum_insured'] >= threshold
        if flagged.sum() > 0:
            df.loc[flagged, 'sum_insured'] = df.loc[flagged, 'sum_insured'] / 1000
            print(f'test_{line}: {flagged.sum()} sum_insured values fixed')

# Construction harmonization
test['buildings']['construction_type'] = test['buildings']['construction_type'].astype(str).str.strip().str.upper().replace(CONSTRUCTION_MAP_EMG)

# Zero-pad motor customer_id
test['motor']['customer_id'] = test['motor']['customer_id'].str.zfill(8)
customers['customer_id'] = customers['customer_id'].astype(str).str.zfill(8)

# Join enrichment
customers_clean = customers.drop(columns=['surname'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df

# === Add cell keys to test data ===
for line in ['motor', 'buildings', 'contents']:
    test[line] = add_cell_key(test[line], line)

# === Compute credibility predictions for test data ===
NEW_TREND = {'motor': 1.04, 'buildings': 1.096, 'contents': 1.049}
cred_test_preds = {}

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    cells = cell_estimates[line].set_index('cell_key')
    # Look up credibility mean for each test policy
    df['cred_mean'] = df['cell_key'].map(cells['cred_mean'])
    # Fallback for unknown cells: use market-level mean
    overall_emg = emg[line]['ultimate_claims'].sum() / emg[line]['exposure'].sum()
    df['cred_mean'] = df['cred_mean'].fillna(overall_emg)
    # Credibility prediction = per-unit-exposure cost × exposure × trend
    cred_test_preds[line] = np.clip(df['cred_mean'] * df['exposure'] * NEW_TREND[line], 0.01, 50000)
    print(f'{line}: cred test pred mean={cred_test_preds[line].mean():.4f}')

# === Compute credibility D2 on emerging training data (in-sample sanity check) ===
print('\n=== CREDIBILITY D2 ON EMERGING TRAINING DATA (in-sample) ===')
cred_train_d2 = {}
for line in ['motor', 'buildings', 'contents']:
    df = emg[line]
    cells = cell_estimates[line].set_index('cell_key')
    df['cred_mean'] = df['cell_key'].map(cells['cred_mean'])
    df['cred_mean'] = df['cred_mean'].fillna(market_ratios[line] * dev[line]['ultimate_claims'].sum() / dev[line]['exposure'].sum())
    cred_pred = np.clip(df['cred_mean'] * df['exposure'], 0.01, 50000)
    d2 = tweedie_d2(df['ultimate_claims'].values, cred_pred)
    cred_train_d2[line] = d2
    print(f'  {line}: D2={d2:.4f}')
print(f'  Avg: {np.mean(list(cred_train_d2.values())):.4f}')

print('\nStep 3 complete.')

In [0]:
# Step 4: Generate credibility-blended submissions

# === Map model predictions to test row_ids per line ===
model_pred_by_row = dict(zip(model_sub['row_id'], model_sub['prediction']))

# === Build blended submissions at different α values ===
alphas = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
submission = sample_submission.copy()

for alpha in alphas:
    for line in ['motor', 'buildings', 'contents']:
        df = test[line]
        cred_p = cred_test_preds[line]
        model_p = df['row_id'].map(model_pred_by_row).fillna(cred_p).values
        model_p = np.clip(model_p, 0.01, 50000)
        blend = np.clip(alpha * cred_p + (1 - alpha) * model_p, 0.01, 50000)
        pred_map = dict(zip(df['row_id'], blend))
        mask = submission['row_id'].isin(pred_map)
        submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
    submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
    sub_name = f'credibility_blend_alpha{alpha:.1f}.csv' if alpha > 0 else 'model_only_v6.csv'
    sub_path = os.path.join(SUB_DIR, sub_name)
    submission.to_csv(sub_path, index=False)
    print(f'  α={alpha:.1f}: saved {sub_name} (mean={submission["prediction"].mean():.4f})')

# === Summary table ===
print('\n=== SUMMARY ===')
print(f'{"Line":<12} {"K":>10} {"R":>8} {"cred_D2":>10}')
for line in ['motor', 'buildings', 'contents']:
    print(f'{line:<12} {cred_K[line]:>10.2f} {market_ratios[line]:>8.4f} {cred_train_d2[line]:>10.4f}')
print(f'\nCredibility avg D2 (in-sample): {np.mean(list(cred_train_d2.values())):.4f}')
print('Model blend D2 (5-fold CV): 0.0642 (from v6 notebook)')
print('\nSubmissions saved to submissions_emerging_v6/')
print('Recommended: try α=0.2 or α=0.3 as a starting point for Kaggle submission')

# Save credibility results
import pickle
cred_results = {
    'cred_K': cred_K, 'market_ratios': market_ratios, 'cred_train_d2': cred_train_d2,
    'cell_estimates': {ln: cell_estimates[ln].to_dict() for ln in cell_estimates},
}
with open(os.path.join(EMG_DIR, 'credibility_results.pkl'), 'wb') as f:
    pickle.dump(cred_results, f)
print(f'\nSaved: {EMG_DIR}credibility_results.pkl')